In [1]:
   !pip install psutil

In [2]:
#collect_sysinfo.py ; output in json format
import json
import psutil
import platform
import socket

import datetime
now = datetime.datetime.now()
print(now)


def collect_system_info():
    data = {}

    # CPU info
    data["cpu"] = {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_processors": psutil.cpu_count(logical=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "cpu_percent_total": psutil.cpu_percent(interval=0.1),
        "cpu_percent_per_cpu": psutil.cpu_percent(interval=0.1, percpu=True),
    }

    # Memory info
    data["memory"] = psutil.virtual_memory()._asdict()
    data["swap"] = psutil.swap_memory()._asdict()

    # Storage info (alle partities)
    storage = {}
    for part in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(part.mountpoint)
            storage[part.mountpoint] = {
                "device": part.device,
                "fstype": part.fstype,
                "opts": part.opts,
                "total": usage.total,
                "used": usage.used,
                "free": usage.free,
                "percent": usage.percent,
            }
        except PermissionError:
            continue
    data["storage"] = storage

    # Network info
    net = {}
    addrs = psutil.net_if_addrs()
    stats = psutil.net_if_stats()
    io = psutil.net_io_counters(pernic=True)
    for iface, addr_list in addrs.items():
        net[iface] = {
            "addrs": [a._asdict() for a in addr_list],
            "stats": stats[iface]._asdict() if iface in stats else None,
            "io": io[iface]._asdict() if iface in io else None,
        }

    data["network"] = {
        "hostname": socket.gethostname(),
        "fqdn": socket.getfqdn(),
        "interfaces": net,
    }

    return data


if __name__ == "__main__":
    info = collect_system_info()
    # JSON output
    print(json.dumps(info, indent=2))

2026-10-07 10:38:28.127986
{
  "cpu": {
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
    "physical_cores": 3,
    "logical_processors": 5,
    "cpu_freq": {
      "current": 2301.0,
      "min": 0.0,
      "max": 2301.0
    },
    "cpu_percent_total": 68.8,
    "cpu_percent_per_cpu": [
      0.0,
      0.0,
      28.6,
      0.0,
      0.0
    ]
  },
  "memory": {
    "total": 7852855296,
    "available": 1676431360,
    "percent": 78.7,
    "used": 6176423936,
    "free": 1676431360
  },
  "swap": {
    "total": 9497038848,
    "used": 1169391616,
    "free": 8327647232,
    "percent": 12.3,
    "sin": 0,
    "sout": 0
  },
  "storage": {
    "C:\\": {
      "device": "C:\\",
      "fstype": "NTFS",
      "opts": "rw,fixed",
      "total": 253507211264,
      "used": 201466695680,
      "free": 52040515584,
      "percent": 79.5
    }
  },
  "network": {
    "hostname": "Unluckydodo",
    "fqdn": "Unluckydodo.hubkah

In [3]:
print(info.keys())
print(info["network"].keys())

dict_keys(['cpu', 'memory', 'swap', 'storage', 'network'])
dict_keys(['hostname', 'fqdn', 'interfaces'])


In [4]:
print("SCHIJVEN:", list(info["storage"].keys()))
print()
for naam, data in info["network"]["interfaces"].items():
    up = data["stats"]["isup"] if data["stats"] else None
    ipv4 = [a["address"] for a in data["addrs"] if a["family"] == 2]
    print(f"{naam:35} up={up}  ipv4={ipv4}")

SCHIJVEN: ['C:\\']

Ethernet 2                          up=True  ipv4=['172.16.150.1']
LAN-verbinding* 1                   up=False  ipv4=['169.254.167.228']
LAN-verbinding* 2                   up=False  ipv4=['169.254.102.185']
Wi-Fi                               up=True  ipv4=['10.109.224.173']
Bluetooth-netwerkverbinding         up=False  ipv4=['169.254.100.225']
Ethernet                            up=False  ipv4=['169.254.145.136']
Loopback Pseudo-Interface 1         up=True  ipv4=['127.0.0.1']


## Opdracht 1 - Structuur van de dictionary

| Gegeven | Pad in de dictionary |
|---|---|
| Computernaam | info["network"]["hostname"] |
| Processor | info["cpu"]["processor"] |
| Fysieke cores | info["cpu"]["physical_cores"] |
| Max CPU-frequentie | info["cpu"]["cpu_freq"]["max"] |
| RAM (bytes) | info["memory"]["total"] |
| Bestandssysteem | info["storage"][schijf]["fstype"] |
| Opslagcapaciteit (bytes) | info["storage"][schijf]["total"] |
| MAC, IPv4, IPv6 | info["network"]["interfaces"][naam]["addrs"] (family -1, 2 en 23) |
| Computermodel | niet aanwezig in het basisscript |

## Opdracht 2 - Criteria

**Primaire netwerkinterface:** de interface waarlangs mijn computer naar het internet gaat. Dat bepaal ik door te kijken welk lokaal IP-adres het systeem gebruikt voor verkeer naar een extern adres (8.8.8.8, er wordt niets verstuurd). Virtuele adapters (zoals Ethernet 2 met 172.16.150.1), loopback en interfaces met een 169.254-adres tel ik niet mee, omdat die geen echte internetverbinding hebben. Op mijn computer is dat Wi-Fi (10.109.224.173).

**Opslag:** alleen de systeemschijf C:\, omdat die op elke Windows-computer bestaat en het besturingssysteem bevat. Mijn computer heeft maar één schijf.

**Eenheden:** RAM en opslag in GB (1 GB = 1024^3 bytes, afgerond op 2 decimalen), CPU-frequentie in MHz. Dit spreek ik af met mijn groepsleden.

In [7]:
import socket, subprocess, platform, psutil, datetime
print(datetime.datetime.now())

GB = 1024**3

def get_model():
    """Computermodel zit niet in psutil, dus vraag het aan het OS."""
    try:
        if platform.system() == "Windows":
            cmd = ["powershell", "-NoProfile", "-Command",
                   "(Get-CimInstance Win32_ComputerSystem).Model"]
        elif platform.system() == "Darwin":
            cmd = ["sysctl", "-n", "hw.model"]
        else:
            cmd = ["cat", "/sys/devices/virtual/dmi/id/product_name"]
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        return out.stdout.strip() or "N/A"
    except Exception:
        return "N/A"

def get_primary_interface(net):
    """Zoek de interface waarlangs het verkeer naar het internet gaat."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        local_ip = s.getsockname()[0]
        s.close()
        for name, data in net["interfaces"].items():
            if any(a["address"] == local_ip for a in data["addrs"]):
                return name, data
    except Exception:
        pass
    for name, data in net["interfaces"].items():
        if not (data["stats"] and data["stats"]["isup"]):
            continue
        for a in data["addrs"]:
            if a["family"] == socket.AF_INET and not a["address"].startswith(("169.254", "127.")):
                return name, data
    return None, None

def filter_info(info):
    net = info["network"]
    name, iface = get_primary_interface(net)

    mac = ipv4 = ipv6 = link_local = "N/A"
    if iface:
        for a in iface["addrs"]:
            addr = a["address"].split("%")[0]
            if a["family"] == psutil.AF_LINK:
                mac = addr
            elif a["family"] == socket.AF_INET:
                ipv4 = addr
            elif a["family"] == socket.AF_INET6:
                if addr.lower().startswith("fe80"):
                    link_local = addr
                elif ipv6 == "N/A":
                    ipv6 = addr

    system_drive = "C:\\" if platform.system() == "Windows" else "/"
    disk = info["storage"].get(system_drive) or next(iter(info["storage"].values()), {})
    freq = psutil.cpu_freq()

    return {
        "hostname":        net["hostname"],
        "model":           get_model(),
        "processor":       info["cpu"]["processor"],
        "physical_cores":  info["cpu"]["physical_cores"],
        "cpu_max_mhz":     round(freq.max, 0) if freq else "N/A",
        "ram_gb":          round(info["memory"]["total"] / GB, 2),
        "fs_type":         disk.get("fstype", "N/A"),
        "storage_gb":      round(disk.get("total", 0) / GB, 2),
        "primary_iface":   name or "N/A",
        "mac":             mac,
        "ipv4":            ipv4,
        "ipv6":            ipv6,
        "ipv6_link_local": link_local,
    }

info = collect_system_info()
filtered = filter_info(info)
for k, v in filtered.items():
    print(f"{k:16} {v}")

2026-10-07 11:20:43.482958
hostname         Unluckydodo
model            HP ProBook 455 G8 Notebook PC
processor        AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
physical_cores   3
cpu_max_mhz      2301.0
ram_gb           7.31
fs_type          NTFS
storage_gb       236.1
primary_iface    Wi-Fi
mac              70-9C-D1-E9-BE-BB
ipv4             10.109.224.173
ipv6             N/A
ipv6_link_local  N/A


In [8]:
for a in info["network"]["interfaces"]["Wi-Fi"]["addrs"]:
    print(a["family"], a["address"])

-1 70-9C-D1-E9-BE-BB
2 10.109.224.173


In [9]:
import json

# 1. Dict omzetten naar een JSON-tekst
json_tekst = json.dumps(filtered, indent=2)
print(type(filtered), "->", type(json_tekst))
print(json_tekst)

# 2. Opslaan in een .json-bestand
bestandsnaam = f"sysinfo_{filtered['hostname']}.json"
with open(bestandsnaam, "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2)
print("Opgeslagen als:", bestandsnaam)

# 3. Opnieuw inlezen als Python dict
with open(bestandsnaam, "r", encoding="utf-8") as f:
    ingelezen = json.load(f)
print(type(ingelezen))

# 4. Vergelijken met het origineel
print("Zijn ze gelijk?", ingelezen == filtered)
for sleutel in ["hostname", "model", "ram_gb", "ipv4", "mac"]:
    print(sleutel, "->", filtered[sleutel], "|", ingelezen[sleutel])

<class 'dict'> -> <class 'str'>
{
  "hostname": "Unluckydodo",
  "model": "HP ProBook 455 G8 Notebook PC",
  "processor": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
  "physical_cores": 3,
  "cpu_max_mhz": 2301.0,
  "ram_gb": 7.31,
  "fs_type": "NTFS",
  "storage_gb": 236.1,
  "primary_iface": "Wi-Fi",
  "mac": "70-9C-D1-E9-BE-BB",
  "ipv4": "10.109.224.173",
  "ipv6": "N/A",
  "ipv6_link_local": "N/A"
}
Opgeslagen als: sysinfo_Unluckydodo.json
<class 'dict'>
Zijn ze gelijk? True
hostname -> Unluckydodo | Unluckydodo
model -> HP ProBook 455 G8 Notebook PC | HP ProBook 455 G8 Notebook PC
ram_gb -> 7.31 | 7.31
ipv4 -> 10.109.224.173 | 10.109.224.173
mac -> 70-9C-D1-E9-BE-BB | 70-9C-D1-E9-BE-BB


## Opdracht 3 - Verschil tussen een Python dict en JSON

Een **Python dict** is een datastructuur in het geheugen van Python. Je kunt er direct mee werken in code, bijvoorbeeld `filtered["ram_gb"]`. Een dict gebruikt Python-syntax: enkele of dubbele quotes, `True`/`False`, `None`, en kan ook andere Python-types bevatten.

**JSON** is een tekstformaat om gegevens op te slaan en uit te wisselen tussen programma's, bestanden en via het internet (bijvoorbeeld API's). Het is gewoon tekst (een `str`) met een vaste syntaxis: alleen dubbele quotes, `true`/`false` en `null`. Python kan er niet rechtstreeks mee rekenen of in zoeken, tot je het met `json.loads()` of `json.load()` omzet naar een dict.

Met `json.dumps()` / `json.dump()` zet ik een dict om naar JSON, met `json.loads()` / `json.load()` weer terug. Na het opslaan en inlezen van `sysinfo_Unluckydodo.json` was de ingelezen dict gelijk aan de originele (`True`).

In [10]:
import csv

kolommen = ["hostname", "model", "processor", "physical_cores", "cpu_max_mhz",
            "ram_gb", "fs_type", "storage_gb", "primary_iface", "mac",
            "ipv4", "ipv6", "ipv6_link_local"]

csv_naam = f"sysinfo_{filtered['hostname']}.csv"

# Schrijven: eerste regel = kolomnamen, tweede regel = deze computer
with open(csv_naam, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=kolommen, delimiter=";")
    writer.writeheader()
    rij = {k: filtered.get(k, "N/A") for k in kolommen}   # ontbrekend = N/A
    writer.writerow(rij)
print("Opgeslagen als:", csv_naam)

# Terugleggen om te controleren
with open(csv_naam, "r", newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    for rij in reader:
        for k, v in rij.items():
            print(f"{k:16} {v}")

Opgeslagen als: sysinfo_Unluckydodo.csv
hostname         Unluckydodo
model            HP ProBook 455 G8 Notebook PC
processor        AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
physical_cores   3
cpu_max_mhz      2301.0
ram_gb           7.31
fs_type          NTFS
storage_gb       236.1
primary_iface    Wi-Fi
mac              70-9C-D1-E9-BE-BB
ipv4             10.109.224.173
ipv6             N/A
ipv6_link_local  N/A


## Opdracht 4 - CSV

**Bestandsnaam:** sysinfo_Unluckydodo.csv (formaat: sysinfo_<hostname>.csv)

**Kolommen:** hostname, model, processor, physical_cores, cpu_max_mhz, ram_gb, fs_type, storage_gb, primary_iface, mac, ipv4, ipv6, ipv6_link_local. Een computer is een rij.

**Keuzes:**
- Scheidingsteken: `;`, omdat Excel in Belgie dat meestal verwacht en komma's in tekst (zoals de processornaam) dan geen probleem geven.
- Encoding: `utf-8-sig`, zodat Excel speciale tekens correct weergeeft.
- Ontbrekende informatie: `N/A` (bijvoorbeeld ipv6, want mijn Wi-Fi heeft geen IPv6-adres).
- Meerdere disks of interfaces: ik neem alleen de systeemschijf (C:\) en de primaire interface, dus per computer blijft het bij een rij.
- Opnieuw inlezen: het script leest het bestand terug met csv.DictReader.

## Opdracht 4 - CSV

**Bestandsnaam:** sysinfo_Unluckydodo.csv (formaat: sysinfo_<hostname>.csv)

**Kolommen:** hostname, model, processor, physical_cores, cpu_max_mhz, ram_gb, fs_type, storage_gb, primary_iface, mac, ipv4, ipv6, ipv6_link_local. Een computer is een rij.

**Keuzes:**
- Scheidingsteken: `;`, omdat Excel in Belgie dat meestal verwacht en komma's in tekst (zoals de processornaam) dan geen probleem geven.
- Encoding: `utf-8-sig`, zodat Excel speciale tekens correct weergeeft.
- Ontbrekende informatie: `N/A` (bijvoorbeeld ipv6, want mijn Wi-Fi heeft geen IPv6-adres).
- Meerdere disks of interfaces: ik neem alleen de systeemschijf (C:\) en de primaire interface, dus per computer blijft het bij een rij.
- Opnieuw inlezen: het script leest het bestand terug met csv.DictReader.

In [11]:
!pip install openpyxl pandas

In [16]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

xlsx_naam = f"sysinfo_{filtered['hostname']}.xlsx"

# Dezelfde gegevens en kolomnamen als in de CSV: lees de CSV in en schrijf naar Excel
df = pd.read_csv(csv_naam, sep=";", encoding="utf-8-sig", keep_default_na=False)
df.to_excel(xlsx_naam, index=False, sheet_name="sysinfo")

# Opmaak: vette kopregel, kolombreedte, bovenste rij vastzetten
wb = load_workbook(xlsx_naam)
ws = wb["sysinfo"]
for cel in ws[1]:
    cel.font = Font(bold=True)
for kolom in ws.columns:
    breedte = max(len(str(c.value)) for c in kolom if c.value is not None)
    ws.column_dimensions[kolom[0].column_letter].width = breedte + 2
ws.freeze_panes = "A2"
wb.save(xlsx_naam)

print("Opgeslagen als:", xlsx_naam)
print(df.T)

Opgeslagen als: sysinfo_Unluckydodo.xlsx
                                                                 0
hostname                                               Unluckydodo
model                                HP ProBook 455 G8 Notebook PC
processor        AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
physical_cores                                                   3
cpu_max_mhz                                                 2301.0
ram_gb                                                        7.31
fs_type                                                       NTFS
storage_gb                                                   236.1
primary_iface                                                Wi-Fi
mac                                              70-9C-D1-E9-BE-BB
ipv4                                                10.109.224.173
ipv6                                                           N/A
ipv6_link_local                                                N/A


In [18]:
import pandas as pd
from openpyxl import load_workbook

# 1. Inhoud teruglezen uit het .xlsx-bestand
df_terug = pd.read_excel(xlsx_naam, sheet_name="sysinfo", keep_default_na=False)
display(df_terug)

# 2. Controleren of de opmaak er echt in zit
ws = load_workbook(xlsx_naam)["sysinfo"]
print("Werkblad:         ", ws.title)
print("Aantal kolommen:  ", ws.max_column)
print("Aantal rijen:     ", ws.max_row, "(1 kopregel + gegevens)")
print("Kopregel vet:     ", ws["A1"].font.bold)
print("Bovenste rij vast:", ws.freeze_panes)
print("Gelijk aan CSV:   ", df_terug.astype(str).equals(df.astype(str)))

,hostname,model,processor,physical_cores,cpu_max_mhz,ram_gb,fs_type,storage_gb,primary_iface,mac,ipv4,ipv6,ipv6_link_local
0,Unluckydodo,HP ProBook 455 G8 Notebook PC,"AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",3,2301,7.31,NTFS,236.1,Wi-Fi,70-9C-D1-E9-BE-BB,10.109.224.173,N/A,N/A


Werkblad:          sysinfo
Aantal kolommen:   13
Aantal rijen:      2 (1 kopregel + gegevens)
Kopregel vet:      True
Bovenste rij vast: A2
Gelijk aan CSV:    False


In [19]:
# Vergelijk de waarden als getallen waar het getallen zijn (2301.0 == 2301)
try:
    pd.testing.assert_frame_equal(df_terug, df, check_dtype=False)
    print("Gelijk aan CSV: True")
except AssertionError as e:
    print("Gelijk aan CSV: False")
    print(e)

Gelijk aan CSV: True


## Opdracht 5 - Spreadsheetbestand

**Formaat:** .xlsx (`sysinfo_Unluckydodo.xlsx`), gemaakt met `pandas` en `openpyxl`, met dezelfde kolomnamen en gegevens als de CSV. Ik heb het bestand in Jupyter teruggelezen: 13 kolommen, 1 gegevensrij, vette kopregel en bovenste rij vastgezet.

**Opmerking:** een eerste vergelijking met de CSV gaf False, omdat de CSV `2301.0` bevat en Excel `2301` opslaat (tekst tegenover getal). Bij vergelijken als getallen zijn de gegevens gelijk (True).

**CSV tegenover XLSX:**
- Een CSV is platte tekst zonder opmaak. Het is klein, universeel en makkelijk door scripts te lezen, maar Excel moet het scheidingsteken en de encoding raden.
- Een XLSX is een echt spreadsheetbestand. Het ondersteunt opmaak, meerdere werkbladen, formules en grafieken, en bewaart getallen als getallen.

**Wanneer wat:** CSV voor eenvoudige uitwisseling en verwerking door scripts (zoals het samenvoegen in opdracht 6). XLSX voor rapporten en overzichten die mensen lezen en verder bewerken.

In [23]:
import glob, os, datetime
import pandas as pd
print(datetime.datetime.now())

# Pas dit pad aan naar de gedeelde map van je groep.
# "." betekent: dezelfde map als dit notebook.
map_met_csv = "."

kolommen = ["hostname", "model", "processor", "physical_cores", "cpu_max_mhz",
            "ram_gb", "fs_type", "storage_gb", "primary_iface", "mac",
            "ipv4", "ipv6", "ipv6_link_local"]

# 1. Alle sysinfo_*.csv zoeken, behalve het resultaatbestand zelf
bestanden = sorted(glob.glob(os.path.join(map_met_csv, "sysinfo_*.csv")))
bestanden = [b for b in bestanden if os.path.basename(b) != "sysinfo_all.csv"]
print("Gevonden bestanden:")
for b in bestanden:
    print("  ", os.path.basename(b))

# 2. Inlezen en controleren
frames = []
for b in bestanden:
    d = pd.read_csv(b, sep=";", encoding="utf-8-sig", keep_default_na=False, dtype=str)
    ontbreekt = [k for k in kolommen if k not in d.columns]
    if ontbreekt:
        print(f"Let op: {os.path.basename(b)} mist kolommen {ontbreekt}, opgevuld met N/A")
    for k in ontbreekt:
        d[k] = "N/A"
    d = d[kolommen].replace("", "N/A")        # lege cellen ook N/A
    frames.append(d)

# 3. Samenvoegen en wegschrijven
totaal = pd.concat(frames, ignore_index=True)
totaal = totaal.drop_duplicates(subset="hostname", keep="last")   # een rij per computer
totaal.to_csv(os.path.join(map_met_csv, "sysinfo_all.csv"),
              sep=";", index=False, encoding="utf-8-sig")
print(f"\nSamengevoegd: {len(totaal)} computer(s) in sysinfo_all.csv")
display(totaal)

2026-10-07 11:40:36.644988
Gevonden bestanden:
   sysinfo_Unluckydodo.csv

Samengevoegd: 1 computer(s) in sysinfo_all.csv


,hostname,model,processor,physical_cores,cpu_max_mhz,ram_gb,fs_type,storage_gb,primary_iface,mac,ipv4,ipv6,ipv6_link_local
0,Unluckydodo,HP ProBook 455 G8 Notebook PC,"AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",3,2301.0,7.31,NTFS,236.1,Wi-Fi,70-9C-D1-E9-BE-BB,10.109.224.173,N/A,N/A


## Opdracht 6 - Samenvoegen

**Afspraken in de groep:**
- Kolommen (zelfde namen en volgorde): hostname, model, processor, physical_cores, cpu_max_mhz, ram_gb, fs_type, storage_gb, primary_iface, mac, ipv4, ipv6, ipv6_link_local
- Eenheden: GB voor RAM en opslag, MHz voor frequentie
- Scheidingsteken `;`, encoding `utf-8-sig`, ontbrekende waarden `N/A`
- Bestandsnaam: `sysinfo_<hostname>.csv`
- Uitwisselen via: (vul zelf in, bijvoorbeeld OneDrive of Teams)

**Hoe het script werkt:**
- `glob` zoekt automatisch alle bestanden `sysinfo_*.csv` (behalve het resultaat zelf).
- Ontbrekende kolommen worden opgevuld met `N/A`, lege cellen ook.
- `pd.concat` voegt alles samen, `drop_duplicates` houdt een rij per computer.
- Het resultaat staat in `sysinfo_all.csv`, zonder manueel kopieren.

**Nieuwe computers later toevoegen:** het CSV-bestand in de map zetten en het script opnieuw uitvoeren.

In [24]:
import subprocess, os, datetime
print(datetime.datetime.now())

def ps(cmd):
    r = subprocess.run(["powershell", "-NoProfile", "-Command", cmd],
                       capture_output=True, text=True, timeout=30)
    return r.stdout.strip()

GB = 1024**3
iface = filtered["primary_iface"]

echt = {
    "hostname":       os.environ.get("COMPUTERNAME", "N/A"),
    "processor":      ps("(Get-CimInstance Win32_Processor | Select -First 1).Name"),
    "physical_cores": ps("(Get-CimInstance Win32_Processor | Measure-Object NumberOfCores -Sum).Sum"),
    "logical_cpus":   ps("(Get-CimInstance Win32_Processor | Measure-Object NumberOfLogicalProcessors -Sum).Sum"),
    "cpu_max_mhz":    ps("(Get-CimInstance Win32_Processor | Select -First 1).MaxClockSpeed"),
    "ram_gb":         round(int(ps("(Get-CimInstance Win32_ComputerSystem).TotalPhysicalMemory")) / GB, 2),
    "ram_ingebouwd_gb": round(int(ps("(Get-CimInstance Win32_PhysicalMemory | Measure-Object Capacity -Sum).Sum")) / GB, 2),
    "storage_gb":     round(int(ps("(Get-CimInstance Win32_LogicalDisk -Filter \"DeviceID='C:'\").Size")) / GB, 2),
    "mac":            ps(f"(Get-NetAdapter -Name '{iface}').MacAddress").upper().replace(":", "-"),
    "ipv4":           ps(f"(Get-NetIPAddress -InterfaceAlias '{iface}' -AddressFamily IPv4).IPAddress"),
}

def gelijk(a, b):
    try:
        return abs(float(a) - float(b)) < 0.5
    except ValueError:
        return str(a).strip().lower() == str(b).strip().lower()

controles = ["hostname", "processor", "physical_cores", "cpu_max_mhz",
             "ram_gb", "storage_gb", "mac", "ipv4"]

print(f"{'veld':16} {'dataset':35} {'echt (Windows)':35} resultaat")
for veld in controles:
    ok = gelijk(filtered[veld], echt[veld])
    print(f"{veld:16} {str(filtered[veld])[:34]:35} {str(echt[veld])[:34]:35} {'OK' if ok else 'VERSCHIL'}")

print("\nExtra info om verschillen te verklaren:")
print("Logische processors (Windows):", echt["logical_cpus"])
print("Ingebouwd RAM (som van de geheugenmodules):", echt["ram_ingebouwd_gb"], "GB")

2026-10-07 11:58:17.749139
veld             dataset                             echt (Windows)                      resultaat
hostname         Unluckydodo                         UNLUCKYDODO                         OK
processor        AMD64 Family 25 Model 80 Stepping   AMD Ryzen 5 5600U with Radeon Grap  VERSCHIL
physical_cores   3                                   3                                   OK
cpu_max_mhz      2301.0                              2301                                OK
ram_gb           7.31                                7.31                                OK
storage_gb       236.1                               236.1                               OK
mac              70-9C-D1-E9-BE-BB                   70-9C-D1-E9-BE-BB                   OK
ipv4             10.109.224.173                      10.109.224.173                      OK

Extra info om verschillen te verklaren:
Logische processors (Windows): 5
Ingebouwd RAM (som van de geheugenmodules): 8.0 GB


## Opdracht 7 - Validatie

| Veld | Dataset | Echt (Windows) | Resultaat |
|---|---|---|---|
| hostname | Unluckydodo | UNLUCKYDODO | OK |
| processor | AMD64 Family 25 Model 80 Stepping 0 | AMD Ryzen 5 5600U with Radeon Graphics | VERSCHIL |
| physical_cores | 3 | 3 | OK |
| cpu_max_mhz | 2301.0 | 2301 | OK |
| ram_gb | 7.31 | 7.31 | OK |
| storage_gb | 236.1 | 236.1 | OK |
| mac | 70-9C-D1-E9-BE-BB | 70-9C-D1-E9-BE-BB | OK |
| ipv4 | 10.109.224.173 | 10.109.224.173 | OK |

**Verklaring verschillen:**
- Processor: platform.processor() geeft de familie-string, Windows geeft de commerciële naam. Dit is normaal.

**Extra info:**
- Logische processors (Windows): 5
- Ingebouwd RAM: 8.0 GB

**Automatische controles die ik kan doen:**
1. Lege waarden controleren (N/A).
2. IP valideren met ipaddress.ip_address().
3. MAC valideren met een regex.
4. Eenheden en bereiken controleren (RAM > 0, MHz > 0).
5. Vergelijken met ipconfig /all via subprocess.

In [2]:
import requests
token = ""
url = "https://webexapis.com/v1/people/me"
headers = {"Authorization": f"Bearer {token}"}
r = requests.get(url, headers=headers)
print(r.status_code)
print(r.json())

200
{'id': 'Y2lzY29zcGFyazovL3VzL1BFT1BMRS83MjdjMDQwZi1hNWNlLTQwZDctYjU0Ni0yNzFmYjVjYzQ5MDE', 'emails': ['doruk.yurdakul@student.odisee.be'], 'phoneNumbers': [], 'displayName': 'Doruk Yurdakul', 'nickName': 'Doruk', 'firstName': 'Doruk', 'lastName': 'Yurdakul', 'userName': 'doruk.yurdakul@student.odisee.be', 'orgId': 'Y2lzY29zcGFyazovL3VzL09SR0FOSVpBVElPTi84MzFlZTA5Ny00MjFiLTQ2Y2EtODIwYi01Y2VjZDU0NDIxMmQ', 'created': '2026-10-07T12:35:50.863Z', 'lastModified': '2026-10-07T12:38:01.099Z', 'timeZone': 'Europe/Brussels', 'status': 'unknown', 'type': 'person'}


In [9]:
import requests
token =""
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

for groep in ["GROUP_YRO_A", "GROUP_YRO_B", "GROUP_YRO_C"]:
    body = {"title": groep}
    r = requests.post("https://webexapis.com/v1/rooms", headers=headers, json=body)
    print(groep, r.status_code)

GROUP_YRO_A 200
GROUP_YRO_B 200
GROUP_YRO_C 200


In [14]:
!pip install webexpythonsdk

In [16]:
Verwijderd: GROUP_YRO_A
Verwijderd: GROUP_YRO_B
Verwijderd: GROUP_YRO_C

In [17]:
from webexpythonsdk import WebexAPI

token = ""
api = WebexAPI(access_token=token)

for room in api.rooms.list():
    if room.title.startswith("GROUP_YRO_"):
        api.rooms.delete(room.id)
        print("Verwijderd:", room.title)

Verwijderd: GROUP_YRO_C
Verwijderd: GROUP_YRO_B
Verwijderd: GROUP_YRO_A
